# Concepts: Merge LM Output

Merges inference results from `work_concepts_lm_output` into `work_concepts`. Enriches raw concept IDs with metadata from `openalex.common.concepts`, builds keywords with tanh-sized dynamic count (2-12).

- **WHEN MATCHED**: overwrites existing (new predictions take priority)
- **WHEN NOT MATCHED**: inserts new predictions

In [ ]:
DECLARE OR REPLACE VARIABLE filter_threshold FLOAT DEFAULT 0.20;
DECLARE OR REPLACE VARIABLE base_mid         FLOAT DEFAULT 5.0;
DECLARE OR REPLACE VARIABLE half_range       FLOAT DEFAULT 6.0;
DECLARE OR REPLACE VARIABLE center_size      INT   DEFAULT 7;
DECLARE OR REPLACE VARIABLE slope            FLOAT DEFAULT 0.05;

MERGE INTO openalex.works.work_concepts AS target
USING (
  WITH common AS (
    SELECT concept_id, wikidata_id, display_name, level
    FROM openalex.common.concepts
  ),

  lm_exploded AS (
    SELECT
      work_id,
      explode(concepts) AS concept,
      created_timestamp
    FROM openalex.works.work_concepts_lm_output
  ),

  joined AS (
    SELECT DISTINCT
      x.work_id,
      x.concept.id AS concept_id,
      x.concept.score AS score,
      c.wikidata_id,
      c.display_name,
      c.level,
      x.created_timestamp
    FROM lm_exploded x
    JOIN common c ON x.concept.id = c.concept_id
  ),

  enriched AS (
    SELECT
      work_id,
      slice(
        array_sort(
          collect_set(
            NAMED_STRUCT(
              'id', concept_id,
              'wikidata', wikidata_id,
              'display_name', display_name,
              'level', level,
              'score', ROUND(CAST(score AS FLOAT), 4)
            )
          ),
          (left, right) -> CASE
            WHEN left.score > right.score THEN -1
            WHEN left.score < right.score THEN 1
            WHEN left.id < right.id THEN -1
            WHEN left.id > right.id THEN 1
            ELSE 0
          END
        ), 1, 40
      ) AS concepts,
      array_sort(
        array_distinct(
          array_compact(
            collect_list(
              IF(level > 1, STRUCT(
                  concat('https://openalex.org/keywords/',
                    regexp_replace(
                      regexp_replace(
                        regexp_replace(replace(lower(display_name), '\'', ''), '\\s*\\([^)]*\\)', ''),
                        '[^^\\p{L}\\p{N}\\./\u2013\\*#]+', '-'
                      ),
                      '(^-+|-+$)', ''
                    )
                  ) AS id,
                  display_name,
                  ROUND(score, 4) AS score
                ),
                NULL
              )
            )
          )
        ),
        (left, right) -> CASE
          WHEN left.score > right.score THEN -1
          WHEN left.score < right.score THEN 1
          WHEN left.id < right.id THEN -1
          WHEN left.id > right.id THEN 1
          ELSE 0
        END
      ) AS keywords_full,
      max(created_timestamp) AS created_datetime
    FROM joined
    GROUP BY work_id
  )

  SELECT
    work_id,
    concepts,
    slice(
      filter(keywords_full, k -> k.score > 0), 1,
      greatest(2, least(12, round(base_mid +
        half_range * tanh((
          size(filter(keywords_full, k -> k.score > filter_threshold)) - center_size) * slope))))
    ) AS keywords,
    created_datetime
  FROM enriched
) AS source
ON target.work_id = source.work_id

WHEN MATCHED THEN UPDATE SET
  target.concepts = source.concepts,
  target.keywords = source.keywords,
  target.source = 'model_v3',
  target.updated_datetime = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
  work_id,
  concepts,
  keywords,
  source,
  created_datetime,
  updated_datetime
) VALUES (
  source.work_id,
  source.concepts,
  source.keywords,
  'model_v3',
  source.created_datetime,
  current_timestamp()
);

In [ ]:
OPTIMIZE openalex.works.work_concepts;